# ResUNet MRI → CT Baseline: L1 + MSE Only

This standalone Kaggle notebook trains a supervised ResUNet using **only two training-loss terms**:

\[
\mathcal L = \lambda_{L1}\,\mathcal L_{L1} + \lambda_{MSE}\,\mathcal L_{MSE},
\qquad \lambda_{L1}=1,\;\lambda_{MSE}=1.
\]

- No SSIM training loss.
- No adversarial/GAN loss.
- No diffusion/noise-prediction loss.
- No anatomical mask is used to mask the training loss or alter image intensities. This is the intended unmasked baseline for a later mask ablation.
- It uses your fixed patient-level split (48 train / 6 validation / 6 test patients). The test set is not evaluated in this notebook and is not used for checkpoint selection.
- New outputs go to `/kaggle/working/resunet_l1_mse_baseline/` so the earlier model checkpoints are kept separate.

**Intensity note:** Images are treated as 8-bit grayscale PNGs and normalized to [-1, 1]. This code does not establish a verified physical PNG-to-HU mapping.

In [ ]:
# CELL 1 — Imports, reproducibility, dataset discovery and fixed patient split
import os
import json
import random
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Dataset root: use expected Kaggle path, with a discovery fallback.
base_dir = "/kaggle/input/datasets/lohithkgowda/synthrad2023-brain-2d-png/synthrad_png"
if not (os.path.isdir(os.path.join(base_dir, "mri")) and os.path.isdir(os.path.join(base_dir, "ct"))):
    candidates = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if os.path.basename(root) == "synthrad_png" and "mri" in dirs and "ct" in dirs:
            candidates.append(root)
    if candidates:
        base_dir = candidates[0]

if not (os.path.isdir(os.path.join(base_dir, "mri")) and os.path.isdir(os.path.join(base_dir, "ct"))):
    raise FileNotFoundError(
        "Could not find folders mri/ and ct/ under the SynthRAD dataset. "
        "Check that the Kaggle dataset is attached and update base_dir."
    )

mri_dir = os.path.join(base_dir, "mri")
ct_dir = os.path.join(base_dir, "ct")

# Your established patient-level split. Do not change for comparisons.
VERIFIED_SPLIT = {
    "train": [
        "1BA001", "1BA005", "1BA012", "1BA014", "1BA022", "1BA032", "1BA040", "1BA058",
        "1BA075", "1BA082", "1BA085", "1BA091", "1BA100", "1BA103", "1BA105", "1BA116",
        "1BA131", "1BA143", "1BA151", "1BA158", "1BA159", "1BA164", "1BA175", "1BA184",
        "1BA185", "1BA189", "1BA201", "1BA206", "1BA220", "1BA222", "1BA227", "1BA239",
        "1BA247", "1BA253", "1BA260", "1BA266", "1BA278", "1BA288", "1BA292", "1BA294",
        "1BA305", "1BA325", "1BA328", "1BA336", "1BA345", "1BA358", "1BA359", "1BA368"
    ],
    "val": ["1BA076", "1BA111", "1BA125", "1BA141", "1BA300", "1BA379"],
    "test": ["1BA054", "1BA097", "1BA172", "1BA234", "1BA256", "1BA307"]
}

# Guard against patient overlap between splits.
assert set(VERIFIED_SPLIT["train"]).isdisjoint(VERIFIED_SPLIT["val"])
assert set(VERIFIED_SPLIT["train"]).isdisjoint(VERIFIED_SPLIT["test"])
assert set(VERIFIED_SPLIT["val"]).isdisjoint(VERIFIED_SPLIT["test"])

# Exact filename pairing: only files present in both directories are used.
valid_ext = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
mri_files = {f for f in os.listdir(mri_dir) if f.lower().endswith(valid_ext) and os.path.isfile(os.path.join(mri_dir, f))}
ct_files = {f for f in os.listdir(ct_dir) if f.lower().endswith(valid_ext) and os.path.isfile(os.path.join(ct_dir, f))}
paired_files = sorted(mri_files.intersection(ct_files))
pat_map = {}
for fname in paired_files:
    pat_id = fname.split("_")[0]
    pat_map.setdefault(pat_id, []).append(fname)
for pat_id in pat_map:
    pat_map[pat_id].sort()

all_split_patients = set(sum(VERIFIED_SPLIT.values(), []))
missing_patients = sorted(pid for pid in all_split_patients if pid not in pat_map)
if missing_patients:
    raise ValueError(f"These fixed-split patients have no matched image files: {missing_patients}")

def get_paths(patient_ids):
    mri_paths, ct_paths = [], []
    for pid in patient_ids:
        for fname in pat_map.get(pid, []):
            mri_paths.append(os.path.join(mri_dir, fname))
            ct_paths.append(os.path.join(ct_dir, fname))
    return mri_paths, ct_paths

tr_m, tr_c = get_paths(VERIFIED_SPLIT["train"])
va_m, va_c = get_paths(VERIFIED_SPLIT["val"])
te_m, te_c = get_paths(VERIFIED_SPLIT["test"])

print(f"Dataset: {base_dir}")
print(f"Train: {len(tr_m)} slices, {len(VERIFIED_SPLIT['train'])} patients")
print(f"Validation: {len(va_m)} slices, {len(VERIFIED_SPLIT['val'])} patients")
print(f"Test: {len(te_m)} slices, {len(VERIFIED_SPLIT['test'])} patients [held out; not used here]")
assert len(tr_m) > 0 and len(va_m) > 0 and len(te_m) > 0

# New, isolated output directory; does not overwrite prior model directories.
OUTPUT_DIR = Path("/kaggle/working/resunet_l1_mse_baseline")
CKPT_DIR = OUTPUT_DIR / "checkpoints"
FIG_DIR = OUTPUT_DIR / "figures"
METRIC_DIR = OUTPUT_DIR / "metrics"
for p in [OUTPUT_DIR, CKPT_DIR, FIG_DIR, METRIC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 128
BATCH_SIZE = 8
NUM_WORKERS = 2
NUM_EPOCHS = 80
BASE_FILTERS = 64
LR = 1e-4
WEIGHT_DECAY = 1e-4
MIN_LR = 1e-6
LAMBDA_L1 = 1.0
LAMBDA_MSE = 1.0
USE_AMP = (device.type == "cuda")

CONFIG = {
    "experiment": "ResUNet_unmasked_L1_plus_MSE",
    "seed": SEED,
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "base_filters": BASE_FILTERS,
    "optimizer": "AdamW",
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "CosineAnnealingLR",
    "min_learning_rate": MIN_LR,
    "lambda_l1": LAMBDA_L1,
    "lambda_mse": LAMBDA_MSE,
    "uses_anatomical_mask_in_loss": False,
    "uses_ssim_loss": False,
    "dataset_root": base_dir,
    "patient_split": VERIFIED_SPLIT,
}
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)


In [ ]:
# CELL 2 — Dataset with paired augmentation; no anatomical masking
class SynthRADUnmaskedDataset(Dataset):
    """Paired grayscale MRI/CT dataset with no mask used in preprocessing or loss."""
    def __init__(self, mri_paths, ct_paths, image_size=128, is_train=False):
        if len(mri_paths) != len(ct_paths):
            raise ValueError("MRI/CT path counts do not match.")
        self.mri_paths = list(mri_paths)
        self.ct_paths = list(ct_paths)
        self.image_size = image_size
        self.is_train = is_train

    def __len__(self):
        return len(self.mri_paths)

    def __getitem__(self, idx):
        mri_path = self.mri_paths[idx]
        ct_path = self.ct_paths[idx]
        mri_arr = np.asarray(Image.open(mri_path).convert("L"), dtype=np.float32)
        ct_arr = np.asarray(Image.open(ct_path).convert("L"), dtype=np.float32)

        # Resize both paired images identically. No MRI/CT mask is applied.
        mri_arr = cv2.resize(mri_arr, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        ct_arr = cv2.resize(ct_arr, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)

        # Map uint8 grayscale values [0, 255] to [-1, 1].
        mri = torch.from_numpy((mri_arr / 255.0 * 2.0 - 1.0)).unsqueeze(0).float()
        ct = torch.from_numpy((ct_arr / 255.0 * 2.0 - 1.0)).unsqueeze(0).float()

        # Paired geometric augmentations: apply the same transform to MRI and CT.
        if self.is_train:
            if random.random() < 0.5:
                mri = TF.hflip(mri)
                ct = TF.hflip(ct)
            angle = random.uniform(-8.0, 8.0)
            tx = int(random.uniform(-4, 4))
            ty = int(random.uniform(-4, 4))
            scale = random.uniform(0.96, 1.04)
            args = dict(
                angle=angle,
                translate=[tx, ty],
                scale=scale,
                shear=[0.0, 0.0],
                interpolation=TF.InterpolationMode.BILINEAR,
                fill=-1.0,
            )
            mri = TF.affine(mri, **args)
            ct = TF.affine(ct, **args)

        patient_id = os.path.basename(mri_path).split("_")[0]
        return {"mri": mri, "ct": ct, "patient_id": patient_id,
                "slice_name": os.path.basename(mri_path)}

train_dataset = SynthRADUnmaskedDataset(tr_m, tr_c, image_size=IMG_SIZE, is_train=True)
val_dataset = SynthRADUnmaskedDataset(va_m, va_c, image_size=IMG_SIZE, is_train=False)

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": (device.type == "cuda"),
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)

# Sanity check one batch before training.
batch = next(iter(train_loader))
print("MRI batch shape:", tuple(batch["mri"].shape))
print("CT batch shape :", tuple(batch["ct"].shape))
print("MRI normalized range:", float(batch["mri"].min()), float(batch["mri"].max()))
print("CT normalized range :", float(batch["ct"].min()), float(batch["ct"].max()))


In [ ]:
# CELL 3 — ResUNet architecture and ONLY L1 + MSE loss
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_groups=8):
        super().__init__()
        groups = num_groups if out_channels % num_groups == 0 else (4 if out_channels % 4 == 0 else 1)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(groups, out_channels)
        self.act1 = nn.LeakyReLU(0.2, inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(groups, out_channels)
        self.act2 = nn.LeakyReLU(0.2, inplace=True)
        self.shortcut = (
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.GroupNorm(groups, out_channels),
            ) if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        residual = self.shortcut(x)
        h = self.act1(self.norm1(self.conv1(x)))
        h = self.norm2(self.conv2(h))
        return self.act2(h + residual)

class ResUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        f = base_filters
        self.enc1 = ResBlock(in_channels, f)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ResBlock(f, f * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ResBlock(f * 2, f * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = ResBlock(f * 4, f * 8)
        self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(ResBlock(f * 8, f * 8), ResBlock(f * 8, f * 8))
        self.up4 = nn.ConvTranspose2d(f * 8, f * 4, 2, stride=2)
        self.dec4 = ResBlock(f * 8 + f * 4, f * 4)
        self.up3 = nn.ConvTranspose2d(f * 4, f * 2, 2, stride=2)
        self.dec3 = ResBlock(f * 4 + f * 2, f * 2)
        self.up2 = nn.ConvTranspose2d(f * 2, f, 2, stride=2)
        self.dec2 = ResBlock(f * 2 + f, f)
        self.up1 = nn.ConvTranspose2d(f, f, 2, stride=2)
        self.dec1 = ResBlock(f + f, f)
        self.final_conv = nn.Conv2d(f, out_channels, 1)
        self.final_act = nn.Tanh()

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        b = self.bottleneck(self.pool4(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final_act(self.final_conv(d1))

class L1MSELoss(nn.Module):
    """The total training objective contains exactly L1 and MSE terms."""
    def __init__(self, lambda_l1=1.0, lambda_mse=1.0):
        super().__init__()
        self.lambda_l1 = lambda_l1
        self.lambda_mse = lambda_mse

    def forward(self, prediction, target):
        l1 = F.l1_loss(prediction, target, reduction="mean")
        mse = F.mse_loss(prediction, target, reduction="mean")
        total = self.lambda_l1 * l1 + self.lambda_mse * mse
        return total, l1.detach(), mse.detach()

model = ResUNet(in_channels=1, out_channels=1, base_filters=BASE_FILTERS).to(device)
criterion = L1MSELoss(lambda_l1=LAMBDA_L1, lambda_mse=LAMBDA_MSE)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ResUNet parameters: {num_params:,}")
print("Training loss: Total = 1.0 * L1 + 1.0 * MSE")
print("No SSIM term; no mask applied to the loss; no GAN or diffusion objective.")


In [ ]:
# CELL 4 — Train, validate, and save best checkpoint (test set remains untouched)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=MIN_LR
)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

BEST_CKPT = CKPT_DIR / "best_resunet_l1_mse.pth"
FINAL_CKPT = CKPT_DIR / "final_resunet_l1_mse.pth"
train_history = []
val_history = []
best_val_loss = float("inf")
best_epoch = 0

print("=" * 88)
print("TRAINING RESUNET WITH L1 + MSE ONLY")
print(f"Epochs={NUM_EPOCHS} | Batch={BATCH_SIZE} | LR={LR} | Image={IMG_SIZE}x{IMG_SIZE}")
print("=" * 88)

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    model.train()
    tr_total_sum = tr_l1_sum = tr_mse_sum = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} train", leave=False):
        mri = batch["mri"].to(device, non_blocking=True)
        ct = batch["ct"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            pred = model(mri)
            loss, l1_value, mse_value = criterion(pred, ct)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        tr_total_sum += float(loss.detach().item())
        tr_l1_sum += float(l1_value.item())
        tr_mse_sum += float(mse_value.item())

    lr_scheduler.step()
    n_train_batches = max(len(train_loader), 1)
    tr_total = tr_total_sum / n_train_batches
    tr_l1 = tr_l1_sum / n_train_batches
    tr_mse = tr_mse_sum / n_train_batches

    model.eval()
    va_total_sum = va_l1_sum = va_mse_sum = 0.0
    with torch.inference_mode():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} val", leave=False):
            mri = batch["mri"].to(device, non_blocking=True)
            ct = batch["ct"].to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                pred = model(mri)
                loss, l1_value, mse_value = criterion(pred, ct)
            va_total_sum += float(loss.item())
            va_l1_sum += float(l1_value.item())
            va_mse_sum += float(mse_value.item())

    n_val_batches = max(len(val_loader), 1)
    va_total = va_total_sum / n_val_batches
    va_l1 = va_l1_sum / n_val_batches
    va_mse = va_mse_sum / n_val_batches
    current_lr = optimizer.param_groups[0]["lr"]

    train_history.append({"epoch": epoch, "total_loss": tr_total, "l1_loss": tr_l1,
                          "mse_loss": tr_mse, "learning_rate": current_lr})
    val_history.append({"epoch": epoch, "total_loss": va_total, "l1_loss": va_l1,
                        "mse_loss": va_mse, "learning_rate": current_lr})

    is_best = va_total < best_val_loss
    if is_best:
        best_val_loss = va_total
        best_epoch = epoch
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_total_loss": best_val_loss,
            "config": CONFIG,
        }, BEST_CKPT)

    elapsed = time.time() - start_time
    marker = " ⭐ BEST SAVED" if is_best else ""
    if epoch == 1 or epoch % 5 == 0 or is_best:
        print(
            f"Epoch [{epoch:02d}/{NUM_EPOCHS}] "
            f"Train: total={tr_total:.5f}, L1={tr_l1:.5f}, MSE={tr_mse:.5f} | "
            f"Val: total={va_total:.5f}, L1={va_l1:.5f}, MSE={va_mse:.5f} | "
            f"LR={current_lr:.2e} | {elapsed:.1f}s{marker}"
        )

    # Keep progress logs even if the notebook session is interrupted.
    pd.DataFrame(train_history).to_csv(METRIC_DIR / "train_history.csv", index=False)
    pd.DataFrame(val_history).to_csv(METRIC_DIR / "validation_history.csv", index=False)

# Save final weights separately from the best validation checkpoint.
torch.save({
    "epoch": NUM_EPOCHS,
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
}, FINAL_CKPT)

print("\nTraining finished.")
print(f"Best epoch: {best_epoch}; best validation total loss: {best_val_loss:.6f}")
print(f"Best checkpoint: {BEST_CKPT}")
print(f"Final checkpoint: {FINAL_CKPT}")
print(f"Metrics folder: {METRIC_DIR}")
print("The test set has not been evaluated or used for model selection.")


In [ ]:
# CELL 5 — Plot loss curves and visualize validation predictions
# Load the BEST checkpoint selected using validation L1 + MSE.
checkpoint = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']}")

train_df = pd.DataFrame(train_history)
val_df = pd.DataFrame(val_history)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, key, title in zip(
    axes,
    ["total_loss", "l1_loss", "mse_loss"],
    ["Total loss (L1 + MSE)", "L1 loss", "MSE loss"]
):
    ax.plot(train_df["epoch"], train_df[key], label="Train")
    ax.plot(val_df["epoch"], val_df[key], label="Validation")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "loss_curves_l1_mse.png", dpi=180, bbox_inches="tight")
plt.show()

# Choose a small set of validation examples; no test examples used.
batch = next(iter(val_loader))
mri = batch["mri"].to(device)
ct = batch["ct"].to(device)
with torch.inference_mode():
    pred = model(mri).cpu().numpy()
mri_np = mri.cpu().numpy()
ct_np = ct.cpu().numpy()
mask_count = min(6, pred.shape[0])

fig, axes = plt.subplots(mask_count, 4, figsize=(14, 3.3 * mask_count), squeeze=False)
for i in range(mask_count):
    mri_show = np.clip((mri_np[i, 0] + 1.0) / 2.0, 0, 1)
    ct_show = np.clip((ct_np[i, 0] + 1.0) / 2.0, 0, 1)
    pred_show = np.clip((pred[i, 0] + 1.0) / 2.0, 0, 1)
    err = np.abs(pred[i, 0] - ct_np[i, 0])

    axes[i, 0].imshow(mri_show, cmap="gray", vmin=0, vmax=1)
    axes[i, 0].set_title("MRI")
    axes[i, 1].imshow(ct_show, cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title("Ground-truth CT")
    axes[i, 2].imshow(pred_show, cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title("ResUNet prediction")
    im = axes[i, 3].imshow(err, cmap="inferno", vmin=0, vmax=2)
    axes[i, 3].set_title("Absolute error (normalized scale)")
    for ax in axes[i]:
        ax.axis("off")

fig.colorbar(im, ax=axes[:, 3].tolist(), fraction=0.025, pad=0.02)
fig.suptitle("ResUNet L1 + MSE — Validation Examples", y=1.002)
plt.tight_layout()
plt.savefig(FIG_DIR / "validation_examples_l1_mse.png", dpi=180, bbox_inches="tight")
plt.show()

print("Saved figures:")
print(" -", FIG_DIR / "loss_curves_l1_mse.png")
print(" -", FIG_DIR / "validation_examples_l1_mse.png")
print("Training and validation CSVs:")
print(" -", METRIC_DIR / "train_history.csv")
print(" -", METRIC_DIR / "validation_history.csv")
